# 02 Spark MLlib 训练

**目标**：用 Spark ML 的 Pipeline 完成特征编码与模型训练。

和 sklearn 的核心差异：
- Spark 要求所有特征先拼成一个 **Vector 列**（`VectorAssembler`）
- 类别特征用 `StringIndexer` 转索引，再 `OneHotEncoder` 编码
- 整个流程也是一个 Pipeline，可以整体保存和加载

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = (SparkSession.builder
         .appName('churn-mllib')
         .master('local[*]')
         .config('spark.sql.shuffle.partitions', '8')
         # 同 01：Spark 4.0 默认 ANSI，cast 'N/A' 会抛错；下面第 5 格要过一遍 cast
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)


Spark 4.0.4


## 1. 载入清洗后的数据

In [3]:
import os
path = '../../data/processed/spark_clean.parquet'
if os.path.exists(path):
    data = spark.read.parquet(path)
else:
    # 上一步没写成功时，退回从 CSV 快速重建
    from pyspark.sql import functions as F
    data = spark.read.option('header', True).option('inferSchema', True) \
        .csv('../../data/raw/customer_churn.csv')
    data = data.toDF(*[c.replace('\ufeff','').strip() for c in data.columns])
    data = data.dropDuplicates(['customer_id']).dropna(subset=['churn'])
    data = data.withColumn('avg_monthly', F.col('monthly_charge')) \
               .withColumn('support_per_year', F.col('support_calls'))

print('样本数:', data.count())
data.select('customer_id', 'contract_type', 'payment_method', 'churn').show(5)

样本数: 1440
+-----------+-------------+--------------+-----+
|customer_id|contract_type|payment_method|churn|
+-----------+-------------+--------------+-----+
|     C01246|     按月付费|        信用卡|    0|
|     C01192|     按月付费|        信用卡|    0|
|     C01050|     按月付费|      电子支票|    0|
|     C00648|     按月付费|      电子支票|    0|
|     C00794|     按月付费|      银行转账|    0|
+-----------+-------------+--------------+-----+
only showing top 5 rows


## 2. 构建特征处理阶段

In [4]:
num_cols = ['age', 'tenure_months', 'monthly_charge', 'total_charge',
            'support_calls', 'is_active', 'avg_monthly', 'support_per_year']
cat_cols = ['contract_type', 'payment_method']

stages = []
for c in num_cols:
    data = data.withColumn(c, data[c].cast('double'))
    data = data.fillna({c: 0.0})

for c in cat_cols:
    data = data.fillna({c: 'unknown'})
    idx = StringIndexer(inputCol=c, outputCol=c + '_idx', handleInvalid='keep')
    enc = OneHotEncoder(inputCol=c + '_idx', outputCol=c + '_vec')
    stages += [idx, enc]

assembler = VectorAssembler(
    inputCols=num_cols + [c + '_vec' for c in cat_cols],
    outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaled',
                        withMean=True, withStd=True)
stages += [assembler, scaler]
print(f'特征阶段: {len(stages)} 个')

特征阶段: 6 个


## 3. 划分数据集并训练两个模型

In [5]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print(f'训练 {train.count()} / 测试 {test.count()}')

evaluator = BinaryClassificationEvaluator(labelCol='churn', metricName='areaUnderROC')
results = {}
fitted = {}

for name, clf in [
    ('LogisticRegression', LogisticRegression(featuresCol='scaled', labelCol='churn', maxIter=50)),
    ('RandomForest', RandomForestClassifier(featuresCol='scaled', labelCol='churn',
                                            numTrees=50, maxDepth=6, seed=42)),
]:
    pipe = Pipeline(stages=stages + [clf])
    model = pipe.fit(train)
    fitted[name] = model          # 留住训练好的 pipeline，下一步要存盘
    pred = model.transform(test)
    auc = evaluator.evaluate(pred)
    acc = pred.filter(pred.churn == pred.prediction).count() / pred.count()
    results[name] = {'AUC': round(auc, 4), 'Accuracy': round(acc, 4)}
    print(f'{name:20s} AUC={auc:.4f}  Acc={acc:.4f}')


训练 1165 / 测试 275
LogisticRegression   AUC=0.9487  Acc=0.8982
RandomForest         AUC=0.9339  Acc=0.8800


## 4. 保存 Pipeline

整个 Pipeline 一起保存，推理时不用再手动拼特征。

In [6]:
import json
best_name = max(results, key=lambda k: results[k]['AUC'])
best_pipe = fitted[best_name]
print('最优模型:', best_name, results[best_name])

os.makedirs('../../models', exist_ok=True)
try:
    best_pipe.write().overwrite().save('../../models/spark_pipeline')
    print('Pipeline 已保存: models/spark_pipeline')
except Exception as e:
    print('保存失败:', type(e).__name__, str(e)[:120])

with open('../../models/spark_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(json.dumps(results, ensure_ascii=False, indent=2))


最优模型: LogisticRegression {'AUC': 0.9487, 'Accuracy': 0.8982}
Pipeline 已保存: models/spark_pipeline
{
  "LogisticRegression": {
    "AUC": 0.9487,
    "Accuracy": 0.8982
  },
  "RandomForest": {
    "AUC": 0.9339,
    "Accuracy": 0.88
  }
}


## 小结

- Spark ML 的 Pipeline 和 sklearn 的思路一致：把编码、缩放、模型串成一条链，整体 fit、整体 save
- 区别在于数据始终是分布式的 DataFrame，`features` 必须是一列 Vector
- `http://localhost:4040` 里能看到训练过程的 Stage 划分和 Shuffle 量